In [0]:
from pyspark.sql.functions import *

def spark_session(name):
    return SparkSession.builder.appName(name).getOrCreate()


def read_df_csv(path,sparksess):
    df = sparksess.read.format('csv').option("header","true").option("delimiter",",").option("inferSchema","true").load(path)
    return df

def read_df_json(path,sparksess):
    df = sparksess.read.format('json').option("multiline","true").load(path)
    return df

def mergedf(df1,df2,path):
    return df1.unionByName(df2,allowMissingColumns=True)

def write_file(df, path, mode="overwrite", format="delta"):
    return df.write.mode(mode).format(format).save(path)

def write_table(df, tablename, mode="overwrite"):
    df.write.mode(mode).format("delta").saveAsTable(tablename)  
       
def read_delta_df(spark,path):
    return spark.read.format("delta").load(path)






In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def standardize_staff(df):
    return (
        df
        .withColumn("shipment_id",F.col("shipment_id").cast("long"))
        .withColumn("age",F.col("age").cast("int"))
        .withColumn("role", F.lower("role"))
        .withColumn("origin_hub_city", F.initcap("hub_location"))
        .withColumn("load_dt", F.current_timestamp())
        .withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
        .withColumn("hub_location", F.initcap("hub_location"))
        .drop("first_name", "last_name")
        .withColumnRenamed("full_name", "staff_full_name")
    )

def scrub_geotag(df):
    return (
        df
        .withColumn("city_name", F.initcap("city_name"))
        .withColumn("masked_hub_location", F.initcap("country"))
    )

def standardize_shipments(df):
    return (
        df
        .withColumn("domain", F.lit("Logistics"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("is_expedited", F.lit(False).cast("boolean"))
        .withColumn("shipment_date", F.to_date("shipment_date", "yy-MM-dd"))
        .withColumn("shipment_cost", F.round("shipment_cost", 2))
        .withColumn("shipment_weight_kg", F.col("shipment_weight_kg").cast("double"))
    )

def enrich_shipments(df):
    return (
        df
        .withColumn("route_segment",
            F.concat_ws("-", "source_city", "destination_city"))
        .withColumn("vehicle_identifier",
            F.concat_ws("_", "vehicle_type", "shipment_id"))
        .withColumn("shipment_year", F.year("shipment_date"))
        .withColumn("shipment_month", F.month("shipment_date"))
        .withColumn("is_weekend",
            F.dayofweek("shipment_date").isin([1,7]))
        .withColumn("is_expedited",
            F.col("shipment_status").isin("IN_TRANSIT", "DELIVERED"))
        .withColumn("cost_per_kg",
            F.round(F.col("shipment_cost") / F.col("shipment_weight_kg"), 2))
        .withColumn("tax_amount",
            F.round(F.col("shipment_cost") * 0.18, 2))
        .withColumn("days_since_shipment",
            F.datediff(F.current_date(), "shipment_date"))
        .withColumn("is_high_value",
            F.col("shipment_cost") > 50000)
    )

def split_columns(df):
    return (
        df
        .withColumn("order_prefix", F.substring("order_id", 1, 3))
        .withColumn("order_sequence", F.substring("order_id", 4, 10))
        .withColumn("ship_year", F.year("shipment_date"))
        .withColumn("ship_month", F.month("shipment_date"))
        .withColumn("ship_day", F.dayofmonth("shipment_date"))
        .withColumn("route_lane",
            F.concat_ws("->", "source_city", "destination_city"))
    )

def mask_name(col):
    return F.concat(
        F.substring(col, 1, 2),
        F.lit("****"),
        F.substring(col, -1, 1)
    )


In [0]:
def join_df(df1,df2,how="inner",on="shipment_id"):#To avoid cartesian/cross join, i am adding some column in the on
    return df1.join(df2, on=on, how=how)

def unionDf(df1,df2):
    return df1.union(df2)
def unionDfSql(spark,view1,view2):    
    returndf=spark.sql(f"select * from view1 union select * from view2")
    return returndf